In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# facebook/dinov2-large (exemplar embeddings) is NOT gated - no extra step.
# =============================================================================

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install -q -U transformers accelerate huggingface_hub supervision opencv-python-headless

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # uses HF_TOKEN from Colab secrets, or paste your token (needs access to facebook/sam3)

print("Environment ready.")


In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time
import itertools

import numpy as np
import pandas as pd
import torch

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor
from transformers import AutoImageProcessor, AutoModel   # DINOv2 exemplar embeddings

# UAV orthomosaics are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)


In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# PIPELINE (only the PROMPT SELECTION differs from k_size_exemplars_no_tiling)
#   for each image:
#     crop every GT Rumex box (square crop, full resolution)
#       -> DINOv2 visual embedding (CLS token, L2-normalised)
#       -> pairwise cosine distance matrix  d(i,j) = 1 - cos(z_i, z_j)
#       -> diversity-based selection: the triple whose CLOSEST pair is as far
#          apart as possible (max-min, see CELL 10)
#       -> 3 GT boxes = 3 exemplar prompts
#     the WHOLE UAV image is downscaled so that its longest side = MAX_DIM,
#     ONE SAM3 forward pass per IMAGE receives those K_EXEMPLARS = 3 GT boxes
#     as POSITIVE box prompts (no text prompt, no negatives),
#     the predicted boxes are scaled back to the original resolution,
#     overlapping duplicates are merged by NMS.
#
# Everything downstream of SAM3 (NMS, matcher, evaluation, AP, confusion
# matrices, tables, plots) is IDENTICAL to k_size_exemplars_no_tiling.
#
# The selection is fully deterministic (no anchors, no random sampling, no
# seeds), so there is exactly ONE run per image.
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "k_diverse_exemplars_no_tiling"   # DINOv2 diversity exemplars, whole image, no tiling
K_EXEMPLARS     = 3           # D1 + D2 + D3
N_EXEMPLARS     = K_EXEMPLARS
USE_TILING      = False       # whole image, no tiles
PROMPT_TYPE     = "diversity_dinov2"

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_NAME}"

# --------------------------- whole-image resize -------------------------------
MAX_DIM = 1024                 # resize cap for SAM3 input (no tiling)

# ------------------- DINOv2 embeddings of the GT crops ------------------------
# The crops are taken from the FULL-RESOLUTION image (not from the MAX_DIM copy),
# so the embedding sees the sharpest available pixels of each plant.
#
#   GT bbox -> +10% fixed context -> square padding -> 224x224 -> DINOv2
#
DINOV2_MODEL_ID   = "facebook/dinov2-large"   # ViT-L/14, 1024-d CLS token
EMBEDDING_FEATURE = "cls"      # CLS token of the last hidden layer, then L2-normalised

CROP_MODE    = "context_pad_square"
CROP_CONTEXT = 0.10            # +10% of the box WIDTH left and right, +10% of the box
                               # HEIGHT top and bottom -> the context box is 1.2 x 1.2
                               # times the GT box. Set 0.0 for a tight box.
CROP_PAD_FILL = (124, 116, 104)  # padding colour of the square canvas = ImageNet mean
                               # (becomes ~0 after DINOv2's normalisation, i.e. neutral
                               # grey rather than a hard black border). Use "edge" to
                               # replicate the border pixels instead.

# The DINOv2 image processor DEFAULTS to "resize shortest edge to 256, then CENTER-CROP
# 224", which would cut the context margin away again. The square canvas is therefore
# resized DIRECTLY to 224x224 with centre cropping switched off.
DINOV2_INPUT_SIZE   = 224
DINOV2_CENTER_CROP  = False

DINOV2_BATCH_SIZE = 16         # crops per forward pass
DINOV2_DTYPE      = torch.float32   # embeddings stay fp32 (distances are compared, not trained)

# --------------------- diversity-based exemplar selection ---------------------
# d(i,j) = 1 - cos(z_i, z_j)  on L2-normalised embeddings
# S* = argmax_{|S|=3} min_{i,j in S, i != j} d(i,j)
DIVERSITY_METRIC       = "cosine"     # distance = 1 - cosine similarity
DIVERSITY_OBJECTIVE    = "max_min"    # maximise the SMALLEST pairwise distance inside the triple
DIVERSITY_EXACT_MAX_GT = 300          # up to this many GT boxes the search is EXHAUSTIVE
                                      # (C(300,3) = 4.5M triples, vectorised, < 1 s);
                                      # above it a greedy farthest-point fallback is used.
                                      # The dataset maxes out around 100 GT boxes per image,
                                      # so in practice the search is always exact.

# --------------------------- SAM3 inference ----------------------------------
# The confidence threshold is FIXED from the start. It is used both
#   (a) inside SAM3 post-processing (SAM3 runs EXACTLY ONCE per image), and
#   (b) as the operating point for precision / recall / F1 / IoU1 / IoU2.
# Because (a) == (b), AP and the operating-point metrics use the same predictions.
CONFIDENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40          # SAM3 mask binarisation (kept as in the E02_1 code)
KEEP_MASKS = False              # masks are NEVER stored (RAM / GPU / disk / CSV / NPZ)

# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
PROMPT_IGNORE_IOU  = 0.50     # held_out mode: an unmatched prediction whose best IoU
                              # with a PROMPT GT box is >= this value is IGNORED
                              # (neither TP nor FP). Same value as EVAL_IOU_THRESHOLD
                              # so a single IoU threshold governs the whole protocol.

# ------------------------------- NMS -----------------------------------------
# Fixed from the start (no confidence x NMS sweep, no "best" selection).
NMS_IOU_THRESHOLD = 0.40

EVALUATION_MODES = ["all_gt", "held_out"]
# all_gt   : every GT box of the image is evaluated (classical evaluation).
# held_out : the 3 GT instances used as prompts (D1, D2, D3) are IGNORED, and so
#            are the predictions that fall on them. Answers "how well does SAM3
#            find the REMAINING Rumex plants after being shown three visually
#            DIFFERENT examples?".

# --------------------------- qualitative plot ---------------------------------
PLOT_EVALUATION_MODE = "all_gt"   # per-image AP50 of this mode selects the image
PLOT_MIN_GT_BOXES    = 7          # the plotted image must have at least this many GT boxes
PLOT_MAX_DISPLAY_DIM = 2048       # display-only downscale (the full image is 8192 px wide)
PLOT_SHOW_SCORES     = True       # write the confidence next to each predicted box

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  results folder : {RESULTS_ROOT}")
print(f"  prompts        : {K_EXEMPLARS} GT boxes chosen by {DIVERSITY_OBJECTIVE} "
      f"{DIVERSITY_METRIC} diversity of {DINOV2_MODEL_ID} {EMBEDDING_FEATURE} embeddings")
print(f"  crops          : GT box +{CROP_CONTEXT:.0%} context -> square padding "
      f"({CROP_PAD_FILL}) -> {DINOV2_INPUT_SIZE}x{DINOV2_INPUT_SIZE}, full resolution, "
      f"center_crop={DINOV2_CENTER_CROP}")
print(f"  tiling         : {USE_TILING} (whole image resized to MAX_DIM={MAX_DIM})")
print(f"  confidence     : {CONFIDENCE_THRESHOLD} (fixed, single inference pass per image)")
print(f"  mask threshold : {MASK_THRESHOLD}")
print(f"  NMS IoU        : {NMS_IOU_THRESHOLD} (fixed)")
print(f"  evaluation     : {EVALUATION_MODES}, IoU >= {EVAL_IOU_THRESHOLD}")
print(f"  masks stored   : {KEEP_MASKS}")


In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results_new1/<EXPERIMENT_NAME>/
#   raw_detections/      pre-NMS detections + GT embeddings distance matrix
#                        (NPZ, one file per image)
#   metrics/             per-image table, experiment summary, pooled dataset AP,
#                        size-group recall
#   confusion_matrices/  CSV + PNG for all_gt and held_out
#   plots/               qualitative GT-vs-prediction figure (CELL 26)
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")
PLOTS_DIR             = os.path.join(RESULTS_ROOT, "plots")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR,
          CONFUSION_MATRIX_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
# area_D1/D2/D3 are the areas of the three SELECTED plants (they are a RESULT of
# the diversity selection here, not its criterion - they are logged so that the
# size profile of diversity prompts can be compared with the S/M/L experiment).
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "Prompt_ID", "Prompt_Type",
    "n_gt", "n_prompt_gt",
    "area_D1_px2", "area_D2_px2", "area_D3_px2", "median_gt_area_px2",
    "min_pairwise_distance", "mean_pairwise_distance",
    "gt_mean_distance", "search_mode",
    "n_detections_pre_nms", "image_width", "image_height",
    "npz_file", "embedding_seconds", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)


In [ ]:
# =============================================================================
# CELL 5 - MODELS: SAM3 (detector) + DINOv2 (exemplar embeddings)
# =============================================================================
# Both models stay resident on the GPU for the whole inference loop (CELL 13):
# per image, DINOv2 first embeds the GT crops, the diverse triple is selected,
# and SAM3 then runs once with those three boxes as prompts.
# DINOv2-large is ~1.2 GB in fp32, which fits next to SAM3 on a 16 GB GPU.
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)

dino_processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL_ID)
dino_model = AutoModel.from_pretrained(DINOV2_MODEL_ID, torch_dtype=DINOV2_DTYPE).to(device)
dino_model.eval()

DINOV2_EMBED_DIM = int(dino_model.config.hidden_size)

print(f"DINOv2 loaded ({DINOV2_MODEL_ID}).")
print("  model device:", next(dino_model.parameters()).device)
print("  model dtype :", next(dino_model.parameters()).dtype)
print("  embedding   :", EMBEDDING_FEATURE, f"({DINOV2_EMBED_DIM}-d)")


In [ ]:
# =============================================================================
# CELL 6 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================
# Unchanged logic from the original E02_1 notebook (it worked); only wrapped
# into functions and given comments, same as E02_2's CELL 7.
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<n>.txt) and a flat one (labels/<n>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])


In [ ]:
# =============================================================================
# CELL 7 - IoU
# =============================================================================
# Unchanged from the original notebook - it was already correct and vectorised.
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


print("IoU helpers ready.")


In [ ]:
# =============================================================================
# CELL 8 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE BUG IN THE OLD CODE (both E02_1 and E02_2 originally shared this bug)
# The old matcher did:
#     best_gt = argmax(IoU[pred])          <- global best, ignoring availability
#     if best_iou >= thr and best_gt free: match
# So if the globally best GT was already taken, the prediction was thrown away
# even when ANOTHER, still free GT had IoU >= threshold. That under-counted TPs
# (recall too low, false positives too high) on images with clustered plants.
#
# THE CORRECT RULE (implemented here, same as E02_2)
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32)  # IoU of the accepted match for every prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool)  # track which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable")  # confidence, high -> low

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # nothing left to match
        # Consider ONLY currently unmatched GT boxes (this is the fix).
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))  # best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")


In [ ]:
# =============================================================================
# CELL 9 - RESIZE HELPER (whole-image, no tiling)
# =============================================================================
# E02_1 sends the WHOLE image to SAM3 in one pass (no tiles). Very large UAV
# orthomosaics are downscaled to MAX_DIM on the longest side purely for SAM3
# input size / speed; predictions are rescaled back to full resolution
# immediately after inference (see CELL 11). Unchanged from the original
# E02_1 code.
# =============================================================================

def resize_for_sam3(img, max_dim=MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale


print(f"Resize helper ready (MAX_DIM={MAX_DIM}).")


In [ ]:
# =============================================================================
# CELL 10 - DIVERSITY-BASED EXEMPLAR SELECTION (D1 / D2 / D3)
# =============================================================================
# This is the ONLY methodological change with respect to k_size_exemplars_no_tiling.
#
# STEP 1  crop every GT Rumex of the image, at FULL resolution:
#
#           GT bbox -> +10% fixed context -> square padding -> 224x224 -> DINOv2
#
#         a) the GT box is enlarged by CROP_CONTEXT (10%) of its own width on the
#            left/right and of its own height on the top/bottom, so every plant is
#            embedded together with a little of its surrounding soil/grass, in
#            proportion to its own size;
#         b) that context box is cut out of the image and PASTED, centred, onto a
#            SQUARE canvas of side max(context_w, context_h) filled with
#            CROP_PAD_FILL. Padding (instead of widening the crop) means the extra
#            area is neutral filler rather than more of the field, so the embedding
#            cannot be driven by whatever happens to lie next to an elongated plant.
#            Parts of the context box that fall OUTSIDE the image are padded the same
#            way, so a plant at the image border stays centred in its canvas;
#         c) the square canvas is resized DIRECTLY to 224x224 (aspect ratio of the
#            context box is preserved by the padding, so nothing is stretched).
#            The DINOv2 processor's default "shortest edge 256 + centre-crop 224" is
#            switched OFF (DINOV2_CENTER_CROP = False), otherwise it would crop the
#            context margin away again.
#
# STEP 2  embed each crop with DINOv2 (CLS token of the last hidden layer),
#         then L2-normalise:  z_i = f(crop_i) / ||f(crop_i)||
#
# STEP 3  pairwise cosine distance
#             d(i,j) = 1 - cos(z_i, z_j) = 1 - <z_i, z_j>      (in [0, 2])
#
# STEP 4  diversity-based selection
#             S* = argmax_{|S| = 3}  min_{i,j in S, i != j} d(i,j)
#         "pick the three plants such that even the CLOSEST pair among the three
#          is still as different as possible" (max-min / maximin dispersion).
#         The search is EXHAUSTIVE over all C(G,3) triples, evaluated vectorised,
#         so the optimum is exact - not a greedy approximation. Triples are
#         enumerated in lexicographic order and the FIRST maximiser is taken,
#         so ties are broken by the lowest GT indices -> fully deterministic.
#         (For G > DIVERSITY_EXACT_MAX_GT a greedy farthest-point fallback keeps
#          the cell from exploding; the dataset never reaches that size.)
#
# ROLE ORDER  D1, D2, D3 = the three selected plants sorted by DECREASING mean
#         distance to the other two selected plants (ties -> lower GT index).
#         D1 is therefore the most "isolated" of the three. The order only
#         affects labels/colours and the order in which the boxes are handed to
#         SAM3; the prompt set itself is order-independent.
#
# Fewer than 3 GT boxes: 1 box -> [D1], 2 boxes -> [D1, D2] (both used),
# exactly like the S / S+L fallback of the size-based experiment.
# =============================================================================

def box_areas(boxes):
    """(N,4) [x1,y1,x2,y2] -> (N,) areas in px^2."""
    b = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


# ----------------------------- STEP 1: cropping -------------------------------
def crop_gt_for_embedding(image, box, context=CROP_CONTEXT, pad_fill=CROP_PAD_FILL):
    """
    GT bbox -> +context -> square padding.  (The resize to 224 is done by the
    DINOv2 processor in embed_gt_crops.)

    image : PIL.Image at ORIGINAL resolution
    box   : [x1, y1, x2, y2] in full-resolution pixels
    Returns a SQUARE PIL.Image (RGB) with the padded context box centred in it.
    """
    img_w, img_h = image.size
    x1, y1, x2, y2 = [float(v) for v in box]
    w = max(1.0, x2 - x1)
    y_h = max(1.0, y2 - y1)

    # (a) +context on every side, proportional to the box itself
    left   = int(np.floor(x1 - context * w))
    right  = int(np.ceil (x2 + context * w))
    top    = int(np.floor(y1 - context * y_h))
    bottom = int(np.ceil (y2 + context * y_h))
    crop_w, crop_h = max(1, right - left), max(1, bottom - top)

    # the part of the context box that actually exists inside the image
    ax1, ay1 = max(left, 0), max(top, 0)
    ax2, ay2 = min(right, img_w), min(bottom, img_h)
    if ax2 <= ax1 or ay2 <= ay1:                    # degenerate box outside the image
        return Image.new("RGB", (DINOV2_INPUT_SIZE, DINOV2_INPUT_SIZE),
                         (124, 116, 104) if pad_fill == "edge" else pad_fill)
    patch_img = image.crop((ax1, ay1, ax2, ay2))

    # (b) paste it, centred, on a square canvas -> padding, never stretching
    side = max(crop_w, crop_h)
    off_x = (side - crop_w) // 2 + (ax1 - left)     # keeps the plant centred even at
    off_y = (side - crop_h) // 2 + (ay1 - top)      # the image border

    if pad_fill == "edge":                          # replicate border pixels
        canvas = patch_img.resize((side, side), Image.NEAREST)
        canvas.paste(patch_img, (off_x, off_y))
    else:
        canvas = Image.new("RGB", (side, side), tuple(pad_fill))
        canvas.paste(patch_img, (off_x, off_y))
    return canvas


# ---------------------- STEP 2: DINOv2 embeddings -----------------------------
@torch.no_grad()
def embed_gt_crops(image, gt_boxes, batch_size=DINOV2_BATCH_SIZE):
    """
    image    : PIL.Image, ORIGINAL resolution
    gt_boxes : (G,4) full-resolution GT boxes
    Returns  : (G, D) float32 array of L2-NORMALISED DINOv2 CLS embeddings.
    Pipeline : GT bbox -> +CROP_CONTEXT -> square padding -> 224x224 -> DINOv2 -> CLS.
    """
    crops = [crop_gt_for_embedding(image, b) for b in np.asarray(gt_boxes).reshape(-1, 4)]
    chunks = []
    for start in range(0, len(crops), batch_size):
        batch = crops[start:start + batch_size]
        # explicit 224x224 resize, NO centre crop -> the +10% context survives
        inputs = dino_processor(
            images=batch,
            do_resize=True,
            size={"height": DINOV2_INPUT_SIZE, "width": DINOV2_INPUT_SIZE},
            do_center_crop=DINOV2_CENTER_CROP,
            return_tensors="pt",
        ).to(dino_model.device)
        if DINOV2_DTYPE != torch.float32:
            inputs["pixel_values"] = inputs["pixel_values"].to(DINOV2_DTYPE)
        outputs = dino_model(**inputs)
        if EMBEDDING_FEATURE == "cls":
            feats = outputs.last_hidden_state[:, 0, :]          # CLS token
        else:
            raise ValueError(f"EMBEDDING_FEATURE={EMBEDDING_FEATURE!r} not implemented.")
        chunks.append(feats.float().cpu().numpy())
        del inputs, outputs, feats

    z = np.concatenate(chunks, axis=0).astype(np.float32)
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return z / norms                                           # ||z_i|| = 1


# ------------------- STEP 3: pairwise cosine distances ------------------------
def cosine_distance_matrix(z):
    """z (G,D) L2-normalised -> (G,G) distances d(i,j) = 1 - <z_i, z_j>, diag = 0."""
    d = 1.0 - (np.asarray(z, dtype=np.float32) @ np.asarray(z, dtype=np.float32).T)
    d = np.clip(d, 0.0, 2.0)
    d = 0.5 * (d + d.T)                    # exact symmetry despite float round-off
    np.fill_diagonal(d, 0.0)
    return d.astype(np.float32)


# --------------- STEP 4: max-min (maximin dispersion) selection ---------------
def _maxmin_exhaustive(d, k):
    """Exact argmax over all C(n,k) subsets of the minimum in-subset distance."""
    n = d.shape[0]
    combos = np.fromiter(itertools.chain.from_iterable(itertools.combinations(range(n), k)),
                         dtype=np.int64).reshape(-1, k)             # lexicographic order
    pair_mins = None
    for a, b in itertools.combinations(range(k), 2):
        dist_ab = d[combos[:, a], combos[:, b]]
        pair_mins = dist_ab if pair_mins is None else np.minimum(pair_mins, dist_ab)
    best = int(np.argmax(pair_mins))        # first maximiser -> lowest indices on ties
    return [int(i) for i in combos[best]], float(pair_mins[best])


def _maxmin_greedy(d, k):
    """Farthest-point fallback for very large G: start from the farthest pair,
    then repeatedly add the point whose minimum distance to the set is largest."""
    n = d.shape[0]
    i, j = np.unravel_index(int(np.argmax(np.triu(d, 1))), d.shape)
    chosen = [int(i), int(j)]
    while len(chosen) < k:
        min_to_set = d[:, chosen].min(axis=1)
        min_to_set[chosen] = -np.inf
        chosen.append(int(np.argmax(min_to_set)))
    sub = d[np.ix_(chosen, chosen)]
    min_pair = float(sub[np.triu_indices(len(chosen), 1)].min())
    return sorted(chosen), min_pair


def select_diverse_exemplars(gt_boxes, embeddings, k=K_EXEMPLARS):
    """
    Input : gt_boxes   (G,4) full-resolution GT boxes of ONE image
            embeddings (G,D) L2-normalised DINOv2 embeddings of the same boxes
    Output: dict with
              'indices'   - GT indices in prompt order (D1, D2, D3)
              'roles'     - ["D1","D2","D3"] (shorter when G < 3)
              'areas'     - area of each selected box (px^2, reported only)
              'distance_matrix'        - (G,G) pairwise cosine distances
              'min_pairwise_distance'  - the maximised objective  min_{i,j in S} d(i,j)
              'mean_pairwise_distance' - mean distance inside the selected triple
              'gt_mean_distance'       - mean distance over ALL GT pairs of the image
              'search_mode'            - "exhaustive" | "greedy" | "all_gt_used"
              'median_area', 'all_areas'
    """
    areas = box_areas(gt_boxes)
    n = len(areas)
    if n == 0:
        raise ValueError("The image has no GT boxes -> no diversity-based exemplars.")

    d = cosine_distance_matrix(embeddings)
    all_pairs = d[np.triu_indices(n, 1)] if n > 1 else np.array([0.0], dtype=np.float32)

    if n <= k:                                   # 1 or 2 GT boxes -> use them all
        indices = list(range(n))
        min_pair = float(all_pairs.min()) if n > 1 else float("nan")
        search_mode = "all_gt_used"
    elif n <= DIVERSITY_EXACT_MAX_GT:
        indices, min_pair = _maxmin_exhaustive(d, k)
        search_mode = "exhaustive"
    else:
        indices, min_pair = _maxmin_greedy(d, k)
        search_mode = "greedy"

    # ---- role order: decreasing mean distance to the other selected plants ----
    if len(indices) > 1:
        sub = d[np.ix_(indices, indices)]
        mean_to_others = sub.sum(axis=1) / (len(indices) - 1)
        order = sorted(range(len(indices)),
                       key=lambda p: (-float(mean_to_others[p]), indices[p]))
        indices = [indices[p] for p in order]
        sub = d[np.ix_(indices, indices)]
        mean_pair = float(sub[np.triu_indices(len(indices), 1)].mean())
    else:
        mean_pair = float("nan")

    roles = [f"D{r + 1}" for r in range(len(indices))]

    return {
        "indices": indices,
        "roles": roles,
        "areas": areas[indices],
        "distance_matrix": d,
        "min_pairwise_distance": float(min_pair),
        "mean_pairwise_distance": float(mean_pair),
        "gt_mean_distance": float(all_pairs.mean()),
        "search_mode": search_mode,
        "median_area": float(np.median(areas)),
        "all_areas": areas,
    }


def format_diversity_prompt_id(selection):
    """e.g. indices [12, 5, 40] with roles D1,D2,D3 -> 'D1:12+D2:5+D3:40'."""
    return "+".join(f"{r}:{i}" for r, i in zip(selection["roles"], selection["indices"]))


# --- self-test on toy embeddings (no GPU / no model needed) --------------------
# 5 unit vectors: 0, 1, 2 are mutually orthogonal (distance 1 to each other),
# 3 and 4 are near-duplicates of 0 and 1. The max-min triple must be {0, 1, 2}.
_toy_z = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.9998, 0.02, 0.0, 0.0],
    [0.02, 0.9998, 0.0, 0.0],
], dtype=np.float32)
_toy_z /= np.linalg.norm(_toy_z, axis=1, keepdims=True)
_toy_boxes = np.array([[0, 0, 10, 10]] * 5, dtype=np.float32)
_sel = select_diverse_exemplars(_toy_boxes, _toy_z)
assert sorted(_sel["indices"]) == [0, 1, 2], _sel["indices"]
assert abs(_sel["min_pairwise_distance"] - 1.0) < 1e-4, _sel["min_pairwise_distance"]
# --- self-test of the crop geometry (no GPU / no model needed) -----------------
_probe = Image.new("RGB", (200, 120), (0, 0, 0))
for _box, _expect in [([50, 40, 90, 60], 48),      # 40x20 box -> 48x24 context -> 48x48 canvas
                      ([0, 0, 20, 20], 24),        # touches the corner -> padded, still square
                      ([190, 110, 200, 120], 12)]: # bottom-right corner
    _c = crop_gt_for_embedding(_probe, _box)
    assert _c.size == (_expect, _expect), (_box, _c.size, _expect)
print(f"Crop pipeline ready: GT bbox -> +{CROP_CONTEXT:.0%} context -> square padding "
      f"-> {DINOV2_INPUT_SIZE}x{DINOV2_INPUT_SIZE} -> DINOv2 ({EMBEDDING_FEATURE}).")

print("Diversity-based exemplar selection ready "
      f"({DIVERSITY_OBJECTIVE}, {DIVERSITY_METRIC}). Toy example:",
      format_diversity_prompt_id(_sel),
      f"| min pairwise distance = {_sel['min_pairwise_distance']:.3f} "
      f"| search = {_sel['search_mode']}")


In [ ]:
# =============================================================================
# CELL 11 - SAM3 WHOLE-IMAGE INFERENCE (no tiling, exemplar boxes as prompts)
# =============================================================================
# This is the E02_1 method, KEPT AS IT IS:
#   - resize whole image to MAX_DIM
#   - scale exemplar GT boxes into that resized space
#   - single SAM3 forward pass, no tiling, no exemplar-strip composition
#     (the exemplar plants are already visible in the same image, so they can
#     be passed directly as positive box prompts -- unlike E02_2, which needs
#     the strip trick because a tile may not contain the exemplar plant)
#   - scale predicted boxes back up to full-res
#
# SAM3 is called ONCE per image (one D1/D2/D3 prompt set) at CONFIDENCE_THRESHOLD
# (0.30, fixed from the start); NMS is applied afterwards, same protocol as E02_2.
# =============================================================================

def _to_numpy(x):
    """Handles tensors (incl. bf16/fp16), lists of tensors, or plain arrays."""
    if torch.is_tensor(x):
        if x.dtype in (torch.bfloat16, torch.float16):
            x = x.float()
        return x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        if len(x) > 0 and torch.is_tensor(x[0]):
            x = [t.float() if t.dtype in (torch.bfloat16, torch.float16) else t for t in x]
            return torch.stack([t.detach().cpu() for t in x]).numpy()
        return np.array(x)
    return np.array(x)


def run_sam3_whole_image(image: Image.Image, exemplar_boxes_fullres: list,
                         threshold: float = CONFIDENCE_THRESHOLD,
                         mask_threshold: float = MASK_THRESHOLD):
    """
    exemplar_boxes_fullres: list of [x1,y1,x2,y2] in the ORIGINAL image's pixel coords.

    Returns: pred_boxes_fullres (np.ndarray Nx4), pred_scores (np.ndarray N),
             all boxes/scores with score >= threshold. Masks are requested from
             SAM3 internally (needed by post-processing) but never stored.
    """
    img_w, img_h = image.size
    image_sam, sam_scale = resize_for_sam3(image, MAX_DIM)

    input_boxes_xyxy = [[c * sam_scale for c in box] for box in exemplar_boxes_fullres]
    input_boxes = [input_boxes_xyxy]
    input_boxes_labels = [[1] * len(exemplar_boxes_fullres)]  # 1 = positive prompt

    inputs = sam3_processor(
        images=image_sam,
        input_boxes=input_boxes,
        input_boxes_labels=input_boxes_labels,
        return_tensors="pt",
    ).to(sam3_model.device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    pred_boxes_np = _to_numpy(results["boxes"]).reshape(-1, 4)
    pred_scores_np = _to_numpy(results["scores"]).reshape(-1)

    # Boxes came back in image_sam's (resized) coordinate space -> rescale to full-res
    if len(pred_boxes_np) > 0:
        pred_boxes_np = pred_boxes_np / sam_scale

    # masks are never kept (KEEP_MASKS = False) -> drop the reference immediately
    del results, outputs, inputs

    return pred_boxes_np, pred_scores_np


print("SAM3 whole-image inference helper ready (no tiling, direct box prompts).")


In [ ]:
# =============================================================================
# CELL 12 - PRE-NMS DETECTION STORAGE
# =============================================================================
# For every run (= one image, one D1/D2/D3 prompt set) we store the detections
#   AFTER SAM3 inference at CONFIDENCE_THRESHOLD, converted to original-image
#   coordinates, but BEFORE NMS.
# The offline evaluation (CELL 14 onwards) therefore never needs SAM3 again.
# Masks are never stored. The DINOv2 distance matrix of the image IS stored
# (G x G float32, a few KB), so the selection can be re-analysed or re-done with
# another objective without re-running DINOv2. The embeddings themselves are not
# stored (1024-d x G would dominate the file and nothing downstream uses them).
# =============================================================================

def run_npz_path(image_id):
    """Path of the NPZ holding the pre-NMS detections of one image."""
    return os.path.join(RAW_DETECTIONS_DIR, f"{safe_filename(image_id)}.npz")


def save_run_detections(image_id, boxes, scores, gt_boxes, selection, image_size):
    """
    Write one run's pre-NMS detections to NPZ. The file is self-contained: it also
    stores the GT boxes, the D1/D2/D3 prompt indices (in prompt order) and the
    pairwise DINOv2 distance matrix, so the whole offline evaluation and the final
    plot can run without re-opening label files or re-embedding the crops.
    """
    path = run_npz_path(image_id)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        prompt_indices=np.array(selection["indices"], dtype=np.int32),   # order: D1, D2, D3
        prompt_roles=np.array(selection["roles"]),                       # ["D1","D2","D3"]
        prompt_areas=np.asarray(selection["areas"], dtype=np.float32),
        median_gt_area=np.array(float(selection["median_area"]), dtype=np.float32),
        distance_matrix=np.asarray(selection["distance_matrix"], dtype=np.float32),
        min_pairwise_distance=np.array(float(selection["min_pairwise_distance"]), dtype=np.float32),
        mean_pairwise_distance=np.array(float(selection["mean_pairwise_distance"]), dtype=np.float32),
        gt_mean_distance=np.array(float(selection["gt_mean_distance"]), dtype=np.float32),
        search_mode=np.array(selection["search_mode"]),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        boxes=np.asarray(boxes, dtype=np.float32).reshape(-1, 4),    # x1,y1,x2,y2 (original img)
        scores=np.asarray(scores, dtype=np.float32).reshape(-1),     # confidence >= 0.30
    )
    return path


def load_run_detections(path):
    """Read one run NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "prompt_indices": z["prompt_indices"].astype(int),
            "prompt_roles": [str(r) for r in z["prompt_roles"]],
            "prompt_areas": z["prompt_areas"].reshape(-1),
            "median_gt_area": float(z["median_gt_area"]),
            "distance_matrix": z["distance_matrix"],
            "min_pairwise_distance": float(z["min_pairwise_distance"]),
            "mean_pairwise_distance": float(z["mean_pairwise_distance"]),
            "gt_mean_distance": float(z["gt_mean_distance"]),
            "search_mode": str(z["search_mode"]),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes": z["boxes"].reshape(-1, 4),
            "scores": z["scores"].reshape(-1),
        }


print("Pre-NMS detection storage ready ->", RAW_DETECTIONS_DIR)


In [ ]:
# =============================================================================
# CELL 13 - MAIN GPU INFERENCE LOOP (DINOv2 -> selection -> SAM3)
# =============================================================================
# FOR EACH IMAGE:
#     open the original image ONCE
#     read its GT boxes ONCE
#     crop every GT box and embed it with DINOv2 (CELL 10)
#     pick the 3 most DIVERSE GT boxes (max-min cosine distance, CELL 10)
#         - deterministic, so there is exactly ONE run per image
#     run SAM3 ONCE over the whole (resized) image, the 3 boxes as direct prompts
#     save the PRE-NMS detections + the distance matrix (NPZ)
#     release the image
#
# Images without any Rumex GT box are skipped, exactly like in the other
# experiments, so every experiment is evaluated on the same set of images.
# NO NMS and NO metric computation happens here - that is all done offline in the
# following cells. The loop is resumable: finished images are listed in
# runs_manifest.csv.
# =============================================================================

# ---- resume support ---------------------------------------------------------
done_images = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    other_types = sorted(set(_m["Prompt_Type"].astype(str)) - {PROMPT_TYPE})
    if other_types:
        raise RuntimeError(
            f"{RUN_MANIFEST_CSV} already holds results for another prompt setting "
            f"{other_types}. Use a new EXPERIMENT_NAME for {PROMPT_TYPE} "
            "(or delete the old results folder) so the runs are not mixed.")
    done_images = set(_m["image_ID"].astype(str))
    print(f"Resuming: {len(done_images)} images already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
image_times = []
n_total_images = len(valid_images)

try:
    for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
        if image_id in done_images:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: already done, skipped.")
            continue
        image_t0 = time.time()

        # ---------------- open the original image exactly once ----------------
        image = Image.open(image_path).convert("RGB")
        img_w, img_h = image.size
        gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
        n_gt = len(gt_boxes)

        if n_gt == 0:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
            image.close(); del image; gc.collect()
            continue

        # ------- DINOv2 embeddings + diversity-based selection (CELL 10) -------
        embed_t0 = time.time()
        gt_embeddings = embed_gt_crops(image, gt_boxes)
        selection = select_diverse_exemplars(gt_boxes, gt_embeddings)
        embed_seconds = time.time() - embed_t0

        exemplar_indices = selection["indices"]
        exemplar_boxes_fullres = [gt_boxes[i].tolist() for i in exemplar_indices]
        prompt_id = format_diversity_prompt_id(selection)

        # ------------------------- SAM3 (unchanged) ----------------------------
        sam_t0 = time.time()
        pred_boxes_np, pred_scores_np = run_sam3_whole_image(
            image, exemplar_boxes_fullres,
            threshold=CONFIDENCE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
        )
        sam_seconds = time.time() - sam_t0

        npz_path = save_run_detections(image_id, pred_boxes_np, pred_scores_np,
                                       gt_boxes, selection, (img_w, img_h))

        areas_by_role = dict(zip(selection["roles"], selection["areas"]))
        run_seconds = time.time() - image_t0
        manifest_writer.writerow({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": image_id,
            "Prompt_ID": prompt_id,
            "Prompt_Type": PROMPT_TYPE,
            "n_gt": n_gt,
            "n_prompt_gt": len(exemplar_indices),
            "area_D1_px2": round(float(areas_by_role["D1"])) if "D1" in areas_by_role else "",
            "area_D2_px2": round(float(areas_by_role["D2"])) if "D2" in areas_by_role else "",
            "area_D3_px2": round(float(areas_by_role["D3"])) if "D3" in areas_by_role else "",
            "median_gt_area_px2": round(float(selection["median_area"])),
            "min_pairwise_distance": round(float(selection["min_pairwise_distance"]), 6),
            "mean_pairwise_distance": round(float(selection["mean_pairwise_distance"]), 6),
            "gt_mean_distance": round(float(selection["gt_mean_distance"]), 6),
            "search_mode": selection["search_mode"],
            "n_detections_pre_nms": int(len(pred_scores_np)),
            "image_width": img_w,
            "image_height": img_h,
            "npz_file": os.path.basename(npz_path),
            "embedding_seconds": round(embed_seconds, 2),
            "inference_seconds": round(run_seconds, 2),
        })
        manifest_file.flush()                  # this image is on disk -> resumable
        n_new_runs += 1

        if len(exemplar_indices) < K_EXEMPLARS:
            print(f"    NOTE: only {n_gt} GT box(es) -> {len(exemplar_indices)} prompt(s) "
                  f"({'+'.join(selection['roles'])}).")

        n_dets = int(len(pred_scores_np))

        # ---------------- release the image ------------------------------------
        image.close()
        del image, gt_embeddings, pred_boxes_np, pred_scores_np
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        image_times.append(time.time() - image_t0)
        avg_per_image = float(np.mean(image_times))
        eta = (n_total_images - img_idx) * avg_per_image
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} | {n_gt} GT box(es) | "
              f"prompts={prompt_id} | min_dist={selection['min_pairwise_distance']:.3f} "
              f"(mean over all GT pairs {selection['gt_mean_distance']:.3f}) | "
              f"pre-NMS detections={n_dets} | DINOv2 {embed_seconds:.1f}s + SAM3 {sam_seconds:.1f}s "
              f"= {run_seconds:.1f}s | avg/image={avg_per_image:.1f}s | "
              f"ETA={eta / 60:.1f} min ({eta / 3600:.2f} h)")
finally:
    manifest_file.close()                      # also closed if the loop crashes

total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new images.")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Pre-NMS detections in: {RAW_DETECTIONS_DIR}")


In [ ]:
# =============================================================================
# CELL 14 - LOAD CACHED PRE-NMS DETECTIONS
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 13, so you can restart the runtime, free the GPU and
# still redo the complete evaluation in a few minutes.
# To run the evaluation only: run CELLS 1-4 and 6-12 (skip CELL 5 and CELL 13),
# then continue from here.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[(manifest["experiment_name"] == EXPERIMENT_NAME) &
                    (manifest["Prompt_Type"].astype(str) == PROMPT_TYPE)].copy()
manifest = manifest.drop_duplicates(subset=["image_ID"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_ID"] = str(row["Prompt_ID"])
    run["Prompt_Type"] = str(row["Prompt_Type"])
    runs.append(run)

runs_by_image = {r["image_ID"]: r for r in runs}

print(f"Loaded {len(runs)} runs (= images) for {EXPERIMENT_NAME} ({PROMPT_TYPE}).")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes:", int(sum(len(r['gt_boxes']) for r in runs)))
print("Images with fewer than K prompts:",
      int(sum(1 for r in runs if len(r["prompt_indices"]) < K_EXEMPLARS)))

In [ ]:
# =============================================================================
# CELL 15 - OFFLINE NMS
# =============================================================================
# Even without tiling, SAM3 can occasionally propose more than one overlapping
# box for the same plant in a single forward pass. NMS keeps the highest-
# scoring box of each overlapping group. The NMS IoU threshold is fixed
# (NMS_IOU_THRESHOLD = 0.40) and applied after inference, so the raw pre-NMS
# detections stay untouched -- same protocol as E02_2 (no tile provenance here
# since there are no tiles to attribute duplicates to).
# =============================================================================

def nms(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep - list of kept indices, highest score first.
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold.
    """
    n = len(boxes)
    if n == 0:
        return []
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep = []
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        order = list(rest[ious <= iou_threshold])
    return keep


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes', 'scores' of the surviving detections SORTED BY
    SCORE (high -> low). Sorting by score means that applying an operating
    confidence threshold later is just a prefix selection.
    """
    keep = np.array(nms(run["boxes"], run["scores"], iou_threshold), dtype=int)
    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "n_pre_nms": int(len(run["scores"])),
    }


print("NMS ready. NMS IoU threshold (fixed):", NMS_IOU_THRESHOLD)


In [ ]:
# =============================================================================
# CELL 16 - EVALUATION CORE: all_gt AND held_out
# =============================================================================
# all_gt   : classical evaluation, every GT box of the image counts.
#
# held_out : the GT instances that were shown to SAM3 as visual prompts are
#            REMOVED from the GT set, and predictions that fall on those prompt
#            plants are IGNORED (they are neither TP nor FP, they simply do not
#            exist for this evaluation). It answers: "after being shown a few
#            examples, how well does SAM3 find the REMAINING Rumex plants?"
#
# ORDER OF OPERATIONS (important, this is the agreed protocol):
#   1. match predictions to the evaluated (non-prompt) GT with the corrected
#      one-to-one matcher at EVAL_IOU_THRESHOLD = 0.50
#   2. every STILL UNMATCHED prediction whose best IoU with a PROMPT GT box is
#      >= PROMPT_IGNORE_IOU (0.50) becomes IGNORED
#   3. whatever is still unmatched is a false positive
#   Prompt GT boxes themselves are never counted as false negatives.
# =============================================================================

STATUS_FP, STATUS_TP, STATUS_IGNORED = 0, 1, 2


def split_gt_for_mode(gt_boxes, prompt_indices, mode):
    """
    Input : all GT boxes of the image, the indices used as visual prompts, mode
    Output: (evaluated_gt_boxes, prompt_gt_boxes)
            all_gt   -> (all boxes, empty)
            held_out -> (non-prompt boxes, prompt boxes)
    """
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if mode == "all_gt":
        return gt_boxes, np.zeros((0, 4), dtype=np.float32)
    is_prompt = np.zeros(len(gt_boxes), dtype=bool)
    prompt_indices = np.asarray(prompt_indices, dtype=int)
    if len(prompt_indices):
        is_prompt[prompt_indices] = True
    return gt_boxes[~is_prompt], gt_boxes[is_prompt]


def evaluate_run_predictions(pred_boxes, pred_scores, eval_gt_boxes, prompt_gt_boxes,
                             eval_iou=EVAL_IOU_THRESHOLD, ignore_iou=PROMPT_IGNORE_IOU):
    """
    Evaluate ONE prediction set against ONE GT set.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP / STATUS_IGNORED per prediction
      TP, FP, FN, n_ignored, n_eval_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of evaluated GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
      valid_for_macro - False when there is no GT left to evaluate (held_out runs
             in which every plant of the image was used as a prompt)
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    eval_gt_boxes = np.asarray(eval_gt_boxes, dtype=np.float32).reshape(-1, 4)
    prompt_gt_boxes = np.asarray(prompt_gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_eval_gt = len(pred_boxes), len(eval_gt_boxes)
    status = np.full(n_pred, STATUS_FP, dtype=np.int8) # start eveything as FP then valid matches TP then leftover predictions on prompt as ignored and everything still FP

    # step 1 - corrected one-to-one matching against the evaluated GT
    match = match_one_to_one(pred_boxes, pred_scores, eval_gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP # Mark matched predictions as TP

    # step 2 - ignore the leftovers that sit on a PROMPT plant
    # Therefore prompt-ignore logic applies only to predictions that failed to match evaluated GT
    if n_pred and len(prompt_gt_boxes):
        leftover = np.where(status == STATUS_FP)[0] # Find predictions that are still FP (unmatched preds)
        if len(leftover):
            best_prompt_iou = compute_iou_matrix(pred_boxes[leftover], prompt_gt_boxes).max(axis=1)
            status[leftover[best_prompt_iou >= ignore_iou]] = STATUS_IGNORED

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())          # step 3 - the rest are FP
    n_ignored = int((status == STATUS_IGNORED).sum())
    fn = int(n_eval_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_eval_gt if n_eval_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_eval_gt) if n_eval_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "TP": tp, "FP": fp, "FN": fn, "n_ignored": n_ignored,
        "n_eval_gt": n_eval_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
        "valid_for_macro": bool(n_eval_gt > 0),
    }


print("Evaluation core ready (modes:", EVALUATION_MODES, ")")

In [ ]:
# =============================================================================
# CELL 17 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# WHY AP IS COMPUTED DIFFERENTLY FROM PRECISION / RECALL / F1
#
#   AP is an area under the precision-recall curve. That curve is produced by
#   walking through ALL detections ordered by confidence. Truncating the
#   detection list at an operating threshold would simply cut the tail off the
#   curve and report a smaller area - which says nothing about model quality.
#   Therefore AP always uses ALL saved predictions with score >= 0.30
#   (CONFIDENCE_THRESHOLD), after NMS.
#
#   Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point: they
#   answer "if I deploy the detector with confidence >= c, what happens?".
#   Those use only the detections that survive the operating threshold.
#   In this experiment the operating threshold IS the inference threshold (0.30),
#   so both use exactly the same predictions - they still answer different
#   questions (ranking quality vs. deployed behaviour).
#
#   Both numbers can appear in the same row - they answer different questions.
# =============================================================================

# just a helper function that converts NumPy boxes into the format expected by Supervision
def make_detections(boxes, scores=None):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all episodes are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0: # Check whether there is any GT
        return float("nan"), float("nan")
    try:
        # Create the Supervision metric (metric calculator, provides all pred and GTs, calculates the AP results)
        # Supervision exposes map50 for IoU=0.50 and map50_95 for IoUs 0.50:0.95
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead,
        # so it is excluded from means instead of dragging them down.
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


# Its job is not to calculate AP yet, Its job is to prepare: pred and GT for one run so they can later be given to compute_ap(...)
# nms_run: contains your detections after NMS
def ap_inputs_for_run(nms_run, eval_gt, prompt_gt):
    """
    Build the (prediction, GT) episode used for AP of ONE run.
    All post-NMS predictions with score >= CONFIDENCE_THRESHOLD are used;
    in held_out mode the predictions that were IGNORED (they belong to prompt
    plants) are removed first, exactly like in the operating-point evaluation.
    """
    ev = evaluate_run_predictions(nms_run["boxes"], nms_run["scores"], eval_gt, prompt_gt) # marks each pred as TP,FP,IGNORED, it cares only about the ignored predictions for AP calculation bcz in held_out mode , predictions corresponding to prompt rumex must be removed before AP
    keep = ev["status"] != STATUS_IGNORED  # Remove ignored predictions
    # Convert remaining predictions to Supervision format
    return (make_detections(nms_run["boxes"][keep], nms_run["scores"][keep]),
            make_detections(eval_gt))


print("AP helpers ready (AP always uses every prediction >= "
      f"{CONFIDENCE_THRESHOLD} after NMS).")

In [ ]:
# =============================================================================
# CELL 18 - OPERATING-POINT EVALUATION
# =============================================================================
# Choose one confidence threshold, remove predictions below it, then send the
# remaining predictions to CELL 16 for evaluation. This is the function that
# gives Precision, Recall, F1, IoU1, and IoU2 at one operating threshold.
# =============================================================================

def evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, confidence_threshold):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low) => contains the detections after NMS            eval_gt / prompt_gt - from split_gt_for_mode
            confidence_threshold - the operating point being tested
    Output: the dict of evaluate_run_predictions for the thresholded predictions.
    """
    keep = nms_run["scores"] >= confidence_threshold # boolean mask where True entries are selected and the False entries are excluded
    return evaluate_run_predictions(nms_run["boxes"][keep], nms_run["scores"][keep],
                                    eval_gt, prompt_gt)


print("Operating configuration (fixed from the start, no sweep):")
print(f"  Confidence threshold = {CONFIDENCE_THRESHOLD:.2f}")
print(f"  NMS IoU threshold    = {NMS_IOU_THRESHOLD:.2f}")

In [ ]:
# =============================================================================
# CELL 19 - PLOTTING HELPERS
# =============================================================================
# Colours used everywhere:
#   yellow = GT Rumex boxes
#   cyan   = D1 exemplar prompt (most isolated of the three in embedding space)
#   lime   = D2 exemplar prompt
#   orange = D3 exemplar prompt
#   red    = predictions
# =============================================================================

COLOR_GT, COLOR_PRED = "yellow", "red"
ROLE_COLORS = {"D1": "cyan", "D2": "lime", "D3": "orange"}
ROLE_NAMES = {"D1": "diverse #1", "D2": "diverse #2", "D3": "diverse #3"}


def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


def _draw_boxes(ax, boxes, color, linewidth=1.5, linestyle="-", labels=None, fontsize=6):
    """Draw [x1,y1,x2,y2] boxes (display coordinates) on a matplotlib axis."""
    for k, (x1, y1, x2, y2) in enumerate(np.asarray(boxes).reshape(-1, 4)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=color, linewidth=linewidth,
                                       linestyle=linestyle))
        if labels is not None:
            ax.text(x1, y1 - 2, labels[k],
                    color="white" if color in ("red",) else "black",
                    fontsize=fontsize, va="bottom", ha="left",
                    bbox=dict(facecolor=color, edgecolor="none", pad=0.8, alpha=0.85))


def _display_copy(image, max_display_dim=PLOT_MAX_DISPLAY_DIM):
    """Downscaled numpy copy of the image + the factor to map full-res boxes onto it."""
    w, h = image.size
    s = min(1.0, max_display_dim / max(w, h))
    disp_w, disp_h = max(1, int(w * s)), max(1, int(h * s))
    display = np.asarray(image.resize((disp_w, disp_h), Image.BILINEAR))
    to_disp = np.array([disp_w / w, disp_h / h, disp_w / w, disp_h / h], dtype=np.float32)
    return display, to_disp


print("Plotting helpers ready.")

In [ ]:
# =============================================================================
# CELL 20 - PER-IMAGE METRICS  (one run = one image)
# =============================================================================
# The diversity choice is deterministic, so every image is evaluated exactly once and
# this single table replaces the run-level + image-level tables of the anchor
# experiments. Everything is evaluated at the fixed configuration.
#   AP50 / AP50_95 : all post-NMS predictions, confidence-ranked
#   P / R / F1 / IoU1 / IoU2 / TP / FP / FN : predictions >= CONFIDENCE_THRESHOLD
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]

image_rows = []
for run in runs:
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    gt = run["gt_boxes"]
    for mode in EVALUATION_MODES:
        eval_gt, prompt_gt = split_gt_for_mode(gt, run["prompt_indices"], mode)

        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        ap50, ap5095 = compute_ap([p_det], [g_det])

        ev = evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, CONFIDENCE_THRESHOLD)
        valid = ev["valid_for_macro"]
        nan = float("nan")

        image_rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": run["image_ID"],
            "Prompt_ID": run["Prompt_ID"],
            "Prompt_Type": run["Prompt_Type"],
            "evaluation_mode": mode,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "nms_iou_threshold": NMS_IOU_THRESHOLD,
            "n_gt": len(gt),
            "n_prompt_gt": int(len(run["prompt_indices"])),
            "median_gt_area_px2": round(run["median_gt_area"]),
            "n_eval_gt": ev["n_eval_gt"],
            "n_predictions_pre_nms": nms_run["n_pre_nms"],
            "n_predictions": ev["n_pred"],
            "n_ignored_predictions": ev["n_ignored"],
            "AP50": ap50 if valid else nan,
            "AP50_95": ap5095 if valid else nan,
            "precision": ev["precision"] if valid else nan,
            "recall": ev["recall"] if valid else nan,
            "F1": ev["F1"] if valid else nan,
            "IoU1": ev["IoU1"] if valid else nan,
            "IoU2": ev["IoU2"] if valid else nan,
            "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
            "valid_for_macro": valid,
        })

image_level_df = pd.DataFrame(image_rows).sort_values(
    ["evaluation_mode", "image_ID"]).reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Per-image metrics: {len(image_level_df)} rows "
      f"({image_level_df['image_ID'].nunique()} images x {len(EVALUATION_MODES)} modes) "
      f"-> {IMAGE_LEVEL_CSV}")
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    print(f"  {mode:9s}: {int(sub['valid_for_macro'].sum())} images valid for macro averaging, "
          f"AP50_mean={sub['AP50'].mean():.4f}, F1_mean={sub['F1'].mean():.4f}")

In [ ]:
# =============================================================================
# CELL 21 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Mean over images and std BETWEEN images of every per-image metric, so every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# it contains.
# =============================================================================

summary_rows = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_exemplars": K_EXEMPLARS,
        "embedding_model": DINOV2_MODEL_ID,
        "embedding_feature": EMBEDDING_FEATURE,
        "selection_rule": f"{DIVERSITY_OBJECTIVE}_{DIVERSITY_METRIC}",
        "crop_mode": CROP_MODE,
        "crop_context": CROP_CONTEXT,
        "use_tiling": USE_TILING,
        "max_dim": MAX_DIM,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
        "n_images": int(sub["image_ID"].nunique()),
        "n_images_valid_for_macro": int(sub["valid_for_macro"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = sub[col].mean()
        row[f"{col}_std"] = sub[col].std()      # spread between images
    summary_rows.append(row)

experiment_summary_df = pd.DataFrame(summary_rows)
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 22 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# This is NOT the mean of the per-image AP values. All images are handed to
# supervision at once, so every detection of the whole dataset is ranked in ONE
# precision-recall curve. Here one episode = one image.
# =============================================================================

dataset_rows = []
for mode in EVALUATION_MODES:
    pred_list, gt_list, images_used = [], [], set()
    for run in runs:
        nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        pred_list.append(p_det)
        gt_list.append(g_det)
        images_used.add(run["image_ID"])
    ap50, ap5095 = compute_ap(pred_list, gt_list)   # one PR curve over ALL images
    dataset_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_images": len(images_used),
        "confidence_used_for_AP": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "dataset_AP50": ap50,
        "dataset_AP50_95": ap5095,
    })
    del pred_list, gt_list
    gc.collect()

dataset_ap_df = pd.DataFrame(dataset_rows)
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print("\nFor comparison, the MEAN of the per-image AP50 values (a different quantity):")
print(image_level_df.groupby("evaluation_mode")["AP50"].mean().to_string())

In [ ]:
# =============================================================================
# CELL 23 - DATASET-LEVEL CONFUSION MATRICES
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
# Counts are pooled over every image at the fixed configuration. In held_out mode
# the D1/D2/D3 prompt plants and the detections that were ignored do not appear.
# =============================================================================

confusion_summary = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    tp, fp, fn = int(sub["TP"].sum()), int(sub["FP"].sum()), int(sub["FN"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = safe_f1(precision, recall)

    cm_df = pd.DataFrame(
        [[tp, fn], [fp, np.nan]],
        index=["actual_rumex", "actual_background"],
        columns=["predicted_rumex", "predicted_background"],
    )
    cm_df.to_csv(os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.csv"))

    plot_confusion_matrix(
        tp, fp, fn,
        f"{EXPERIMENT_NAME} - {mode}\ndiversity prompts | conf={CONFIDENCE_THRESHOLD:.2f}, "
        f"NMS IoU={NMS_IOU_THRESHOLD:.2f}, eval IoU={EVAL_IOU_THRESHOLD:.2f}",
        os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.png"))

    confusion_summary.append({
        "experiment_name": EXPERIMENT_NAME, "evaluation_mode": mode,
        "TP": tp, "FP": fp, "FN": fn,
        "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
        "confidence_threshold": CONFIDENCE_THRESHOLD, "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    })
    print(f"{mode:9s}: TP={tp}  FP={fp}  FN={fn}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

confusion_summary_df = pd.DataFrame(confusion_summary)
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)
print("\nConfusion matrices saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 24 - RECALL PER SIZE GROUP (how do diversity prompts fare across plant sizes?)
# =============================================================================
# For every image the GT boxes are split at that image's MEDIAN area:
#   "area <= median" (the smaller half) and "area > median" (the larger half).
# Matching uses all_gt at the fixed operating point (same matcher as everywhere).
# The prompt plants themselves are reported separately, because SAM3 usually
# re-detects them and they would flatter the recall of their size group.
# NOTE: size is NOT the selection criterion in this experiment - this table shows
# whether three VISUALLY DIVERSE prompts happen to cover both size halves anyway,
# and is directly comparable with the same table of k_size_exemplars_no_tiling.
# =============================================================================

size_rows, prompt_rows = [], []
for run in runs:
    gt = run["gt_boxes"]
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    match = match_one_to_one(nms_run["boxes"][keep], nms_run["scores"][keep], gt,
                             EVAL_IOU_THRESHOLD)
    found = match["gt_match_pred"] >= 0

    areas = box_areas(gt)
    is_prompt = np.isin(np.arange(len(gt)), run["prompt_indices"])
    small_half = areas <= run["median_gt_area"]
    for name, grp in [("area <= median", small_half), ("area >  median", ~small_half)]:
        mask = grp & ~is_prompt                      # non-prompt GT boxes only
        size_rows.append({
            "image_ID": run["image_ID"], "size_group": name,
            "n_gt_non_prompt": int(mask.sum()), "found": int((found & mask).sum()),
            "recall": float((found & mask).sum() / mask.sum()) if mask.sum() else float("nan"),
        })
    for role, idx in zip(run["prompt_roles"], run["prompt_indices"]):
        prompt_rows.append({"image_ID": run["image_ID"], "role": role,
                            "gt_index": int(idx), "area_px2": float(areas[idx]),
                            "re_detected": bool(found[idx])})

size_group_df = pd.DataFrame(size_rows)
prompt_hits_df = pd.DataFrame(prompt_rows)
size_group_df.to_csv(os.path.join(METRICS_DIR, "size_group_recall.csv"), index=False)
prompt_hits_df.to_csv(os.path.join(METRICS_DIR, "prompt_plants_redetected.csv"), index=False)

pooled = (size_group_df.groupby("size_group")[["n_gt_non_prompt", "found"]].sum()
          .assign(recall_pooled=lambda d: d["found"] / d["n_gt_non_prompt"]))
per_image_mean = size_group_df.groupby("size_group")["recall"].mean().rename("recall_mean_over_images")

print("Recall of NON-PROMPT GT boxes per size group (all_gt matching):")
print(pooled.join(per_image_mean).to_string())
print("\nPrompt plants re-detected (share per role):")
print(prompt_hits_df.groupby("role")["re_detected"].agg(["count", "sum", "mean"]).to_string())
print("\nSaved:", os.path.join(METRICS_DIR, "size_group_recall.csv"), "and",
      os.path.join(METRICS_DIR, "prompt_plants_redetected.csv"))

In [ ]:
# =============================================================================
# CELL 25 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Prompts per image        : {K_EXEMPLARS} GT boxes selected by DINOv2 embedding "
      f"diversity (max-min cosine distance), no text, no negatives")
print(f"Embedding model          : {DINOV2_MODEL_ID} ({EMBEDDING_FEATURE} token, L2-normalised)")
print(f"Crops                    : {CROP_MODE} crop, context {CROP_CONTEXT:.2f}, full resolution")
print(f"Tiling                   : {USE_TILING}  (whole image, longest side -> {MAX_DIM}px)")
print(f"Confidence threshold     : {CONFIDENCE_THRESHOLD:.2f} (fixed; SAM3 run once per image)")
print(f"Mask threshold           : {MASK_THRESHOLD:.2f}")
print(f"NMS IoU threshold        : {NMS_IOU_THRESHOLD:.2f} (fixed)")
print(f"Evaluation IoU           : {EVAL_IOU_THRESHOLD:.2f}")
print(f"Images                   : {len(runs)}")
_min_d = np.array([r["min_pairwise_distance"] for r in runs], dtype=float)
_all_d = np.array([r["gt_mean_distance"] for r in runs], dtype=float)
print(f"Prompt diversity         : min pairwise distance mean={np.nanmean(_min_d):.3f} "
      f"(median={np.nanmedian(_min_d):.3f}), mean distance over ALL GT pairs="
      f"{np.nanmean(_all_d):.3f}")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["evaluation_mode", "AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["evaluation_mode", "dataset_AP50", "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["evaluation_mode", "TP", "FP", "FN",
                            "precision_micro", "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} image NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)


In [ ]:
# =============================================================================
# CELL 26 - QUALITATIVE PLOT: BEST IMAGE, GT (left) vs PREDICTIONS (right)
# =============================================================================
# IMAGE SELECTION (per-image AP50, mode = PLOT_EVALUATION_MODE):
#   1. keep only images that have at least PLOT_MIN_GT_BOXES (7) GT boxes
#      and take the one with the highest AP50
#   2. if NO image has 7 GT boxes: keep the images with the HIGHEST number of GT
#      boxes and take the one among them with the highest AP50
#   ties are broken by F1, then by the number of GT boxes.
#
# Right panel = post-NMS predictions of that image with
# score >= CONFIDENCE_THRESHOLD, plus its D1 / D2 / D3 prompts (dashed).
# The image is downscaled for DISPLAY only (PLOT_MAX_DISPLAY_DIM).
# =============================================================================

def select_image_for_plot(mode=PLOT_EVALUATION_MODE, min_gt=PLOT_MIN_GT_BOXES):
    """Output: (per-image row, selection_rule_text) or (None, reason)."""
    img_df = image_level_df[(image_level_df["evaluation_mode"] == mode) &
                            image_level_df["AP50"].notna()].copy()
    if img_df.empty:
        return None, "no image with a valid AP50"

    candidates = img_df[img_df["n_gt"] >= min_gt]
    if len(candidates):
        rule = (f"highest AP50 among the {len(candidates)} images "
                f"with >= {min_gt} GT boxes")
    else:
        max_gt = int(img_df["n_gt"].max())
        candidates = img_df[img_df["n_gt"] == max_gt]
        rule = (f"no image has >= {min_gt} GT boxes -> highest AP50 among "
                f"the {len(candidates)} image(s) with the most GT boxes ({max_gt})")

    ranked = candidates.sort_values(["AP50", "F1", "n_gt"], ascending=False, na_position="last")
    print("Top candidates:")
    print(ranked[["image_ID", "n_gt", "Prompt_ID", "n_predictions", "AP50", "F1",
                  "precision", "recall"]].head(5).to_string(index=False))
    return ranked.iloc[0], rule


def plot_best_image(mode=PLOT_EVALUATION_MODE):
    row, rule = select_image_for_plot(mode)
    if row is None:
        print("Nothing to plot:", rule)
        return
    plot_image_id = row["image_ID"]
    run = runs_by_image[plot_image_id]

    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    pred_boxes, pred_scores = nms_run["boxes"][keep], nms_run["scores"][keep]
    gt_boxes_plot = run["gt_boxes"]

    # ---- open the image, downscale for display only ---------------------------
    image_paths = {r[3]: r[1] for r in valid_images}
    if plot_image_id not in image_paths:
        print("Image file not found for", plot_image_id)
        return
    with Image.open(image_paths[plot_image_id]) as im:
        display, to_disp = _display_copy(im.convert("RGB"))

    # ---- figure: GT (left) | predictions (right) -------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(22, 8.5))
    for ax in axes:
        ax.imshow(display)
        ax.axis("off")

    gt_d = gt_boxes_plot * to_disp
    _draw_boxes(axes[0], gt_d, COLOR_GT, linewidth=1.5)
    axes[0].set_title(f"Ground truth: {len(gt_boxes_plot)} Rumex boxes", fontsize=12)

    score_labels = [f"{v:.2f}" for v in pred_scores] if PLOT_SHOW_SCORES else None
    _draw_boxes(axes[1], pred_boxes * to_disp, COLOR_PRED, linewidth=1.5, labels=score_labels)
    for role, idx in zip(run["prompt_roles"], run["prompt_indices"]):
        _draw_boxes(axes[1], gt_d[[idx]], ROLE_COLORS[role], linewidth=2.5, linestyle="--")
    axes[1].set_title(
        f"Predictions: {len(pred_boxes)} boxes | prompts {run['Prompt_ID']}\n"
        f"AP50={row['AP50']:.3f}  P={row['precision']:.3f}  R={row['recall']:.3f}  "
        f"F1={row['F1']:.3f}  TP={int(row['TP'])} FP={int(row['FP'])} FN={int(row['FN'])}",
        fontsize=12)

    legend_handles = [
        Line2D([0], [0], color=COLOR_GT, lw=2, label="ground truth"),
        Line2D([0], [0], color=COLOR_PRED, lw=2, label="prediction"),
    ] + [Line2D([0], [0], color=ROLE_COLORS[r], lw=2.5, linestyle="--",
                label=f"{r} prompt ({ROLE_NAMES[r]})") for r in run["prompt_roles"]]
    fig.legend(handles=legend_handles, loc="lower center", ncol=len(legend_handles),
               fontsize=11, frameon=False)
    fig.suptitle(
        f"{EXPERIMENT_NAME} | {plot_image_id} | mode={mode}\n"
        f"diversity prompts {run['Prompt_ID']} | min pairwise cosine distance "
        f"{run['min_pairwise_distance']:.3f} | no tiling (MAX_DIM={MAX_DIM}px) | "
        f"conf={CONFIDENCE_THRESHOLD:.2f}, NMS IoU={NMS_IOU_THRESHOLD:.2f}, "
        f"mask thr={MASK_THRESHOLD:.2f}\nselection: {rule}",
        fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.90])

    png_path = os.path.join(PLOTS_DIR, f"best_image_{safe_filename(plot_image_id)}_{mode}.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"\nSelected image : {plot_image_id} ({len(gt_boxes_plot)} GT boxes)")
    print(f"Selection rule : {rule}")
    print(f"Prompts        : {run['Prompt_ID']}")
    print(f"Figure saved   : {png_path}")


plot_best_image(PLOT_EVALUATION_MODE)